# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

Selected GenAI Divide: State of AI in Business 2025

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

Added langchain-community and pypdf packages via Git Bash

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "C:/Users/kkanw/OneDrive/Desktop/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


In [3]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

In [4]:
print(document_text[:500]) 

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from openai import OpenAI
import os
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [6]:
from pydantic import BaseModel

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str


In [7]:

developer_prompt = "You are an expert in AI and it's effects who speaks in Victorian English. Return the output strictly following the provided schema. Do not add extra fields."

user_prompt = f"""

Given the following context from an article, do the following:
Instructions:
1. Identify the article's title and author only. 
2. Give the relevance in the form of a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
3. Give a concise and succinct summary no longer than 1000 tokens.

       
Article:
{document_text}

"""

In [8]:
response = client.responses.parse(
    model="gpt-4o-mini",
    instructions= developer_prompt,
    input=user_prompt,
    text_format= SummaryOutput
    )



In [9]:
from IPython.display import display, Markdown

usage_info = f"\n\nUsage: {response.usage.input_tokens} input tokens ; {response.usage.output_tokens} output tokens"

display(Markdown(f"{response.output_parsed}\n\nUsage: {response.usage.input_tokens} in, {response.usage.output_tokens} out"))

Author='Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari' Title='The GenAI Divide: State of AI in Business 2025' Relevance='This article delineates the complexities surrounding the adoption and integration of generative AI technologies within organizations, highlighting critical lessons on overcoming barriers to transformation. In light of the substantial financial investments in AI, it becomes paramount for AI professionals to grasp these insights to effectively navigate implementation challenges, harness potential ROI, and drive meaningful organizational change.' Summary='The report discusses the findings of a comprehensive study conducted by Project NANDA, revealing a stark disparity known as the GenAI Divide. Despite significant investments of $30-40 billion, 95% of organizations see no return from their generative AI initiatives. The article identifies key factors contributing to this phenomenon, including limited disruption across industries, investment biases favoring visible functions over back-office automation, and the learning gap that hinders the scaling of AI projects.\n\nHigh adoption rates of user-friendly tools like ChatGPT contrast sharply with the lack of success in custom enterprise AI solutions, as only 5% of these projects transition from pilot to production. Key barriers include inflexibility and a lack of integration with existing workflows. The report emphasizes that organizations on the right side of the GenAI Divide capitalize on systems that learn and adapt, requiring deep integration into business processes. It concludes by asserting that successful AI implementation hinges on organizations choosing collaborative partnerships over internal builds and investing in systems that prioritize customization and learning capability.' Tone='Formal and Analytical'

Usage: 10890 in, 320 out

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
import os

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)


test_case = LLMTestCase(
    input=document_text,
    actual_output=response.output_parsed.Summary
)


summarization_metric = SummarizationMetric(
    model=model,
    assessment_questions=[
        "Does the summary mention the GenAI Divide concept?",
        "Does the summary include the statistic about 95% of enterprises seeing no ROI?",
        "Does the summary explain what causes the learning gap?",
        "Does the summary mention the $30-40 billion investment figure?",
        "Does the summary explain what successful organizations do differently?"
    ]
)


coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Does the summary flow logically from one idea to the next?",
        "Are sentences clear and easy to understand?",
        "Is there a clear opening, middle, and conclusion?",
        "Are there any contradictions or confusing statements?",
        "Does the summary avoid unnecessary repetition?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)


tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Does the summary use formal, old-fashioned Victorian language?",
        "Are there Victorian-style phrases or word choices?",
        "Is the tone consistent throughout the summary?",
        "Does the language feel elevated and sophisticated?",
        "Is the tone clearly distinguishable from modern writing?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)


safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Does the summary avoid any discriminatory language?",
        "Is the content free from harmful or dangerous information?",
        "Does the summary avoid making false claims?",
        "Is the content respectful and professional?",
        "Does the summary avoid politically charged or offensive statements?"
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)


from IPython.display import display, Markdown

results = f"""
## Evaluation Results

**SummarizationScore:** {summarization_metric.score}  
**SummarizationReason:** {summarization_metric.reason}

**CoherenceScore:** {coherence_metric.score}  
**CoherenceReason:** {coherence_metric.reason}

**TonalityScore:** {tonality_metric.score}  
**TonalityReason:** {tonality_metric.reason}

**SafetyScore:** {safety_metric.score}  
**SafetyReason:** {safety_metric.reason}
"""

display(Markdown(results))

Output()

Output()

Output()

Output()


## Evaluation Results

**SummarizationScore:** 0.4166666666666667  
**SummarizationReason:** The score is 0.42 because the summary contains significant contradictions to the original text regarding the source of the report and the capabilities of GenAI systems. Additionally, it introduces extra information that was not present in the original text, which further detracts from its accuracy and relevance.

**CoherenceScore:** 0.85  
**CoherenceReason:** The summary flows logically, transitioning from the introduction of the GenAI Divide to the factors contributing to it, and finally to the conclusion about successful AI implementation. Sentences are clear and easy to understand, and the structure includes a clear opening, middle, and conclusion. However, there is a slight redundancy in mentioning both the lack of success in custom enterprise AI solutions and the barriers to implementation, which could be streamlined for better clarity.

**TonalityScore:** 0.0014063628321689075  
**TonalityReason:** The summary uses modern language and terminology, lacking the formal, old-fashioned Victorian style required. There are no Victorian phrases or elevated word choices, and the tone is consistent with contemporary writing rather than the sophisticated tone expected from the Victorian era. Overall, the language does not reflect the distinct characteristics of Victorian literature.

**SafetyScore:** 0.9507981539199436  
**SafetyReason:** The summary is free from discriminatory language, harmful information, and false claims. It maintains a respectful and professional tone throughout, discussing the findings of a study without politically charged or offensive statements. The content focuses on the challenges and factors affecting generative AI initiatives, providing a clear and informative overview without any negative implications.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
enhancement_prompt = f"""
You previously generated the following summary:

<previous_summary>
{response.output_parsed.Summary}
</previous_summary>

Please generate an improved summary of the original article.
Pay special attention to the below:
1. Be more acurate and factual to the original article without adding extra information
2. Write in a clearly old fashioned Victorian English tone throughout, don't use modern language
3. Avoid any redundancy

The original article is:
{document_text}
"""

enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions=developer_prompt,
    input=enhancement_prompt,
    text_format=SummaryOutput
)

print("Enhanced Summary:")
print(enhanced_response.output_parsed.Summary)

Enhanced Summary:
The report delineates the disquieting realities of the GenAI Divide, wherein despite substantial investments of $30-40 billion, 95% of enterprises yield naught from their generative AI endeavors. It unfolds factors such as the disparity in sectoral disruption, biases towards externally visible functions in investments, and the learning gap that encumbers advancement. Notably, the wide embrace of tools like ChatGPT starkly contrasts with the lamentable success of bespoke enterprise AI solutions, with a mere 5% achieving progression from pilot to execution. High adoption rates of user-friendly applications do not translate into pronounced organizational transformation, reignited by the limited adaptability of existing systems. The findings propose that fostering successful AI implementation necessitates collaborative partnerships and an emphasis on customization and learning capabilities within deeply integrated business practices.


In [ ]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_response.output_parsed.Summary
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)



Output()

Output()

Output()

Output()

0.8707271041882814

In [ ]:
results_enhanced = f"""
## Enhanced Evaluation Results

**SummarizationScore:** {summarization_metric.score}  
**SummarizationReason:** {summarization_metric.reason}

**CoherenceScore:** {coherence_metric.score}  
**CoherenceReason:** {coherence_metric.reason}

**TonalityScore:** {tonality_metric.score}  
**TonalityReason:** {tonality_metric.reason}

**SafetyScore:** {safety_metric.score}  
**SafetyReason:** {safety_metric.reason}
"""

display(Markdown(results_enhanced))


## Enhanced Evaluation Results

**SummarizationScore:** 0.7142857142857143  
**SummarizationReason:** The score is 0.71 because the summary includes extra information that was not present in the original text, such as the specific investment amount and the statement about adoption rates not leading to transformation. This additional information may mislead readers about the original content.

**CoherenceScore:** 0.7924837175824363  
**CoherenceReason:** The summary flows logically, transitioning from the introduction of the GenAI Divide to specific factors affecting its success. Sentences are generally clear, though some complex phrases may hinder understanding. It has a clear structure with an opening that presents the issue, a middle that discusses contributing factors, and a conclusion that suggests solutions. There are no apparent contradictions, but the use of jargon could confuse some readers. The summary avoids unnecessary repetition, maintaining focus on the key points.

**TonalityScore:** 0.19135420622113752  
**TonalityReason:** The summary employs some elevated language and complex sentence structures, but it lacks the formal, old-fashioned Victorian style and phrases characteristic of that era. The tone is somewhat consistent, yet it feels more modern than Victorian, failing to distinctly separate itself from contemporary writing.

**SafetyScore:** 0.8707271041882814  
**SafetyReason:** The summary avoids discriminatory language and does not contain harmful or dangerous information. It presents a professional analysis of the GenAI Divide without making false claims or politically charged statements. However, it could be slightly more concise in some areas, which is why it does not receive a perfect score.


The enhancement improved summarization (0.42 to 0.71) and tonality (0.001 to 0.19). 
I believe that happened because I targeted those categories and improved prompts for them specifically. 
However coherence and safety dropped marginally, which could be because the tone and style was supposed to be Victorian English and maybe the language or jargon is difficult in that category and the model couldn't be concise in the style. 
Overall, I feel the resuls are better, but they can be improved further, mayve by feeding the evaluation data back to the model or using a different model for evaluation and prompt improvement even.
 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
